May return to this notebook periodically for ongoing optimization.

In [1]:
import pandas as pd

Import Round one statistics - These are basic models with no GridSearch optimization and no feature selection. We will average them out, see which features are least correlated with errors, and use those figures to select a new feature set.

In [3]:
df_round1_linreg = pd.read_csv('models_daily_results/Round_1_Linreg.csv')
df_round1_rfr = pd.read_csv('models_daily_results/Round_1_RFR.csv')

In [4]:
dfs_to_merge = df_round1_linreg, df_round1_rfr

In [5]:
df_round1_rfr

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,close_vs_open_pct,...,close_lag1,close_lag2,return_lag1,next_close,next_close_change,next_close_change_pct,pred_next_close_change_pct,pred_error,abs_pred_error,directionally_correct
0,2025-01-02,AA,38.165,39.0400,37.9000,37.99,2703886.0,38.3283,33119.0,-0.004585,...,37.78,37.15,0.016958,35.71,-2.28,-6.001579,-3.40,2.601579,2.601579,1
1,2025-01-03,AA,37.950,37.9500,35.3752,35.71,7491799.0,35.9337,63814.0,-0.059025,...,37.99,37.78,0.005558,36.49,0.78,2.184262,1.80,-0.384262,0.384262,1
2,2025-01-06,AA,36.000,37.0750,35.9200,36.49,5983917.0,36.4794,58230.0,0.013611,...,35.71,37.99,-0.060016,36.24,-0.25,-0.685119,-1.43,-0.744881,0.744881,1
3,2025-01-07,AA,36.890,37.3000,35.7550,36.24,3381466.0,36.2093,39434.0,-0.017620,...,36.49,35.71,0.021843,36.00,-0.24,-0.662252,-1.32,-0.657748,0.657748,1
4,2025-01-08,AA,35.780,36.0300,34.7500,36.00,3775303.0,35.5672,41012.0,0.006149,...,36.24,36.49,-0.006851,35.91,-0.09,-0.250000,-1.16,-0.910000,0.910000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99245,2025-12-24,ZTS,123.100,125.6899,123.0600,125.49,2369020.0,125.1484,29115.0,0.019415,...,123.54,123.78,-0.001939,126.23,0.74,0.589688,0.65,0.060312,0.060312,1
99246,2025-12-26,ZTS,125.160,126.3200,124.8000,126.23,3226934.0,125.8979,45295.0,0.008549,...,125.49,123.54,0.015784,125.98,-0.25,-0.198051,-0.70,-0.501949,0.501949,1
99247,2025-12-29,ZTS,126.160,126.8500,125.5607,125.98,4465924.0,126.0785,51439.0,-0.001427,...,126.23,125.49,0.005897,126.41,0.43,0.341324,0.50,0.158676,0.158676,1
99248,2025-12-30,ZTS,125.550,127.6000,125.4500,126.41,3230541.0,126.5667,45221.0,0.006850,...,125.98,126.23,-0.001981,125.82,-0.59,-0.466735,-0.72,-0.253265,0.253265,1


In [6]:
df_round1_rfr.columns

Index(['timestamp', 'ticker', 'open', 'high', 'low', 'close', 'volume', 'vwap',
       'transactions', 'close_vs_open_pct', 'daily_high_vs_low_pct',
       'prev_day_return', 'SMA10', 'SMA20', 'SMA50', 'SMA100', 'RSI', 'MACD',
       'MACD_signal', 'MACD_hist', 'rolling_volatility', 'close_lag1',
       'close_lag2', 'return_lag1', 'next_close', 'next_close_change',
       'next_close_change_pct', 'pred_next_close_change_pct', 'pred_error',
       'abs_pred_error', 'directionally_correct'],
      dtype='object')

In [11]:
from functools import reduce
import pandas as pd

# Load datasets
df_round1_linreg = pd.read_csv('models_daily_results/Round_1_Linreg.csv')
df_round1_rfr = pd.read_csv('models_daily_results/Round_1_RFR.csv')


# Create dictionary of dataframes to merge
dfs_to_merge = {
    "RFR": df_round1_rfr,
    "Linreg": df_round1_linreg,
}

merge_cols = ["timestamp", "ticker"]

pred_cols = [
    "pred_next_close",
    "pred_next_close_change_pct",
    "pred_error",
    "abs_pred_error",
    "directionally_correct",
]

def prepare_model_df(df, model_name):

    df = df.copy()

    rename_map = {
        col: f"{model_name}_{col}"
        for col in pred_cols
        if col in df.columns
    }

    df = df.rename(columns=rename_map)

    model_cols = list(rename_map.values())

    return df[merge_cols + model_cols]

# Use first dataframe as base
first_key = list(dfs_to_merge.keys())[0]

base_df = dfs_to_merge[first_key].copy()

# Remove original un-prefixed prediction columns
base_df = base_df.drop(
    columns=[col for col in pred_cols if col in base_df.columns]
)

# Create model-specific dfs
model_dfs = [
    prepare_model_df(df, model_name)
    for model_name, df in dfs_to_merge.items()
]

# Merge everything
merged_results = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on=merge_cols,
        how="inner"
    ),
    [base_df] + model_dfs
)

print("Merged shape:", merged_results.shape)

merged_results.head()

Merged shape: (99250, 35)


,timestamp,ticker,open,high,low,close,volume,vwap,transactions,close_vs_open_pct,...,next_close_change,next_close_change_pct,RFR_pred_next_close_change_pct,RFR_pred_error,RFR_abs_pred_error,RFR_directionally_correct,Linreg_pred_next_close_change_pct,Linreg_pred_error,Linreg_abs_pred_error,Linreg_directionally_correct
0,2025-01-02,AA,38.165,39.040,37.9000,37.99,2703886.0,38.3283,33119.0,-0.004585,...,-2.28,-6.001579,-3.40,2.601579,2.601579,1,-1.20,4.801579,4.801579,1
1,2025-01-03,AA,37.950,37.950,35.3752,35.71,7491799.0,35.9337,63814.0,-0.059025,...,0.78,2.184262,1.80,-0.384262,0.384262,1,0.38,-1.804262,1.804262,1
2,2025-01-06,AA,36.000,37.075,35.9200,36.49,5983917.0,36.4794,58230.0,0.013611,...,-0.25,-0.685119,-1.43,-0.744881,0.744881,1,-0.00,0.685119,0.685119,0
3,2025-01-07,AA,36.890,37.300,35.7550,36.24,3381466.0,36.2093,39434.0,-0.017620,...,-0.24,-0.662252,-1.32,-0.657748,0.657748,1,-0.11,0.552252,0.552252,1
4,2025-01-08,AA,35.780,36.030,34.7500,36.00,3775303.0,35.5672,41012.0,0.006149,...,-0.09,-0.250000,-1.16,-0.910000,0.910000,1,0.00,0.250000,0.250000,0


In [13]:
# Find all matching model result columns
pred_error_cols = [
    col for col in merged_results.columns
    if col.endswith("_pred_error")
]

abs_pred_error_cols = [
    col for col in merged_results.columns
    if col.endswith("_abs_pred_error")
]

direction_cols = [
    col for col in merged_results.columns
    if col.endswith("_directionally_correct")
]

# Create aggregate dataframe
model_row_aggs = merged_results.copy()

model_row_aggs["avg_pred_error"] = model_row_aggs[pred_error_cols].mean(axis=1)

model_row_aggs["avg_abs_pred_error"] = model_row_aggs[abs_pred_error_cols].mean(axis=1)

model_row_aggs["pct_directionally_correct"] = (
    model_row_aggs[direction_cols].mean(axis=1) * 100
)

model_row_aggs

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,close_vs_open_pct,...,RFR_pred_error,RFR_abs_pred_error,RFR_directionally_correct,Linreg_pred_next_close_change_pct,Linreg_pred_error,Linreg_abs_pred_error,Linreg_directionally_correct,avg_pred_error,avg_abs_pred_error,pct_directionally_correct
0,2025-01-02,AA,38.165,39.0400,37.9000,37.99,2703886.0,38.3283,33119.0,-0.004585,...,2.601579,2.601579,1,-1.20,4.801579,4.801579,1,3.701579,3.701579,100.0
1,2025-01-03,AA,37.950,37.9500,35.3752,35.71,7491799.0,35.9337,63814.0,-0.059025,...,-0.384262,0.384262,1,0.38,-1.804262,1.804262,1,0.000000,1.094262,100.0
2,2025-01-06,AA,36.000,37.0750,35.9200,36.49,5983917.0,36.4794,58230.0,0.013611,...,-0.744881,0.744881,1,-0.00,0.685119,0.685119,0,0.342560,0.715000,50.0
3,2025-01-07,AA,36.890,37.3000,35.7550,36.24,3381466.0,36.2093,39434.0,-0.017620,...,-0.657748,0.657748,1,-0.11,0.552252,0.552252,1,0.276126,0.605000,100.0
4,2025-01-08,AA,35.780,36.0300,34.7500,36.00,3775303.0,35.5672,41012.0,0.006149,...,-0.910000,0.910000,1,0.00,0.250000,0.250000,0,0.125000,0.580000,50.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99245,2025-12-24,ZTS,123.100,125.6899,123.0600,125.49,2369020.0,125.1484,29115.0,0.019415,...,0.060312,0.060312,1,0.41,-0.179688,0.179688,1,0.030156,0.120000,100.0
99246,2025-12-26,ZTS,125.160,126.3200,124.8000,126.23,3226934.0,125.8979,45295.0,0.008549,...,-0.501949,0.501949,1,-0.14,0.058051,0.058051,1,0.029026,0.280000,100.0
99247,2025-12-29,ZTS,126.160,126.8500,125.5607,125.98,4465924.0,126.0785,51439.0,-0.001427,...,0.158676,0.158676,1,0.21,-0.131324,0.131324,1,0.079338,0.145000,100.0
99248,2025-12-30,ZTS,125.550,127.6000,125.4500,126.41,3230541.0,126.5667,45221.0,0.006850,...,-0.253265,0.253265,1,-0.33,0.136735,0.136735,1,0.068368,0.195000,100.0


In [15]:
# Columns NOT to include in feature correlation analysis
exclude_cols = [
    "timestamp",
    "ticker",

    # Targets / aggregate metrics
    "avg_pred_error",
    "avg_abs_pred_error",
    "pct_directionally_correct",

    # Individual model prediction outputs
]

# Remove all model-specific prediction columns automatically
exclude_cols.extend([
    col for col in model_row_aggs.columns
    if (
        "_pred_" in col
        or "_abs_" in col
        or "_directionally_correct" in col
    )
])

# Candidate feature columns
feature_cols = [
    col for col in model_row_aggs.columns
    if col not in exclude_cols
]

# Correlation to avg absolute prediction error
abs_error_corrs = pd.DataFrame({
    "feature": feature_cols,
    "corr_to_avg_abs_pred_error": [
        model_row_aggs[col].corr(
            model_row_aggs["avg_abs_pred_error"]
        )
        for col in feature_cols
    ]
})

# Correlation to directional correctness %
direction_corrs = pd.DataFrame({
    "feature": feature_cols,
    "corr_to_pct_directionally_correct": [
        model_row_aggs[col].corr(
            model_row_aggs["pct_directionally_correct"]
        )
        for col in feature_cols
    ]
})

# Sort by strongest relationships
abs_error_corrs = abs_error_corrs.sort_values(
    "corr_to_avg_abs_pred_error",
    key=lambda x: x.abs(),
    ascending=False
)

direction_corrs = direction_corrs.sort_values(
    "corr_to_pct_directionally_correct",
    key=lambda x: x.abs(),
    ascending=False
)

print("Top correlations to average absolute prediction error")
display(abs_error_corrs.head(20))

print("\nTop correlations to directional correctness %")
display(direction_corrs.head(20))

Top correlations to average absolute prediction error


,feature,corr_to_avg_abs_pred_error
8,daily_high_vs_low_pct,0.371149
18,rolling_volatility,0.301911
24,next_close_change_pct,0.234871
4,volume,0.199544
6,transactions,0.056373
9,prev_day_return,0.047804
14,RSI,-0.038057
21,return_lag1,0.025291
17,MACD_hist,-0.022705
16,MACD_signal,0.013508



Top correlations to directional correctness %


,feature,corr_to_pct_directionally_correct
8,daily_high_vs_low_pct,-0.178260
18,rolling_volatility,-0.153218
12,SMA50,0.146309
13,SMA100,0.146309
11,SMA20,0.145763
10,SMA10,0.145373
19,close_lag1,0.145291
0,open,0.145276
20,close_lag2,0.145177
2,low,0.145144


In [16]:
# Build combined feature correlation dataframe
feature_corrs = pd.DataFrame({
    "feature": feature_cols,

    "corr_to_avg_abs_pred_error": [
        model_row_aggs[col].corr(
            model_row_aggs["avg_abs_pred_error"]
        )
        for col in feature_cols
    ],

    "corr_to_pct_directionally_correct": [
        model_row_aggs[col].corr(
            model_row_aggs["pct_directionally_correct"]
        )
        for col in feature_cols
    ]
})

# Reverse sign so:
# positive = GOOD
feature_corrs["good_abs_error_corr"] = (
    -feature_corrs["corr_to_avg_abs_pred_error"]
)

# Keep directional correctness positive
feature_corrs["good_direction_corr"] = (
    feature_corrs["corr_to_pct_directionally_correct"]
)

# Weighted composite score
feature_corrs["feature_quality_score"] = (
    (feature_corrs["good_abs_error_corr"] * 0.7) +
    (feature_corrs["good_direction_corr"] * 0.3)
)

# Sort best to worst
feature_corrs = feature_corrs.sort_values(
    "feature_quality_score",
    ascending=False
)

feature_corrs.head(30)

,feature,corr_to_avg_abs_pred_error,corr_to_pct_directionally_correct,good_abs_error_corr,good_direction_corr,feature_quality_score
2,low,-0.011529,0.145144,0.011529,0.145144,0.051614
12,SMA50,-0.010160,0.146309,0.010160,0.146309,0.051005
13,SMA100,-0.010160,0.146309,0.010160,0.146309,0.051005
22,next_close,-0.010233,0.145013,0.010233,0.145013,0.050667
3,close,-0.010093,0.144966,0.010093,0.144966,0.050555
5,vwap,-0.010015,0.145036,0.010015,0.145036,0.050522
20,close_lag2,-0.009881,0.145177,0.009881,0.145177,0.050470
19,close_lag1,-0.009810,0.145291,0.009810,0.145291,0.050455
0,open,-0.009765,0.145276,0.009765,0.145276,0.050418
11,SMA20,-0.009278,0.145763,0.009278,0.145763,0.050224


In [14]:
row_agg_cols = [
    "timestamp",
    "ticker",
    "avg_pred_error",
    "avg_abs_pred_error",
    "pct_directionally_correct",
]

model_row_aggs_summary = model_row_aggs[row_agg_cols].copy()

model_row_aggs_summary.head()

,timestamp,ticker,avg_pred_error,avg_abs_pred_error,pct_directionally_correct
0,2025-01-02,AA,3.701579,3.701579,100.0
1,2025-01-03,AA,0.000000,1.094262,100.0
2,2025-01-06,AA,0.342560,0.715000,50.0
3,2025-01-07,AA,0.276126,0.605000,100.0
4,2025-01-08,AA,0.125000,0.580000,50.0
...,...,...,...,...,...
99245,2025-12-24,ZTS,0.030156,0.120000,100.0
99246,2025-12-26,ZTS,0.029026,0.280000,100.0
99247,2025-12-29,ZTS,0.079338,0.145000,100.0
99248,2025-12-30,ZTS,0.068368,0.195000,100.0
